## Установка зависимостей/ядра
Без этого проект может не запуститься

1. `cd ~/old_home/querulus-main`
2. `python3 -m venv .venv`
3. `source .venv/bin/activate`
4. `python -m pip install -U pip setuptools wheel`
5. `pip install outboxml -e . ipykernel pandas numpy pyarrow scikit-learn catboost optuna matplotlib seaborn plotly kaleido environs pymssql openpyxl`
6. `python -m ipykernel install --user --name=querulus --display-name="Python (querulus)"`


# OutBoxML: модель 2 (TARGET_FREQ / TARGET_SEV)

Конфигурации DSM — из артефактов `train_loop_new` (признаки и HPO). В прод идут сырые `predict_proba` / `predict` (без калибровки).

**Данные:** Hive `models.querulus_df_final_3` или кэш `data/processed/df_final_3.parquet`.

**Периоды** (см. таблицу `periods` после сборки конфигов)
- **Train (parity):** период `parity_train`.
- **Test:** весь контрольный период test.
- **Test_prod:** 15% самых поздних дат Test (отсечка `prod_cutoff`).

**Порог τ frequency:** подбирается в `collect` (блок B); в `example` только загрузка из артефактов `train_loop_new/`.

**Prod-refit:** обучение на train ∪ 85% Test до `prod_cutoff`; оценка финэффекта на Test_prod при том же τ из collect.

После prod-refit: FactorsPlot, cohort, `EMailDSResult`, экспорт pickle/parquet/meta. На сервисе — `querulus_dq_bounds_{version}.json` (`apply_frozen_dq_bounds`).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
OUTBOXML_ROOT = PROJECT_ROOT.parent.parent
for _p in (SRC, OUTBOXML_ROOT, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
print("PROJECT_ROOT", PROJECT_ROOT)
print("OUTBOXML_ROOT", OUTBOXML_ROOT)


In [ ]:
import json
import pickle
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format

from outboxml.core.prepared_datasets import PrepareDataset
from querulus.training.email_report import QuerulusEMailDSResult
from outboxml.data_subsets import DataPreprocessor
from outboxml.datasets_manager import DataSetsManager
from outboxml.export_results import ResultExport

from querulus.training.build_outboxml_configs import (
    dataframe_for_dsm,
    default_model_version,
    ensure_legacy_inflation_column,
    prepare_datasets_from_config,
    ensure_predictable_model,
    unwrap_estimator,
    write_outboxml_configs,
)
from querulus.training.outboxml_metrics import display_dsm_collect_metrics
from querulus.features.data_quality import write_service_dq_bounds
from querulus.fin_effect import (
    create_summary_table,
    export_business_html,
    print_best_threshold_report,
    resolve_fin_effect_config,
    run_fin_effect_pipeline,
)


In [ ]:
MODEL_VERSION = default_model_version(business="2", increment="v1")
HIVE_TABLE = "models.querulus_df_final_3"
# USE_SYNTHETIC=True — только синтетика (Hive не запрашивается; файл создаётся при отсутствии).
# USE_SYNTHETIC=False — только Hive или df_final_3.parquet; без синтетики, иначе FileNotFoundError.
USE_SYNTHETIC = False
LOCAL_PARQUET_DEFAULT = PROJECT_ROOT / "data" / "processed" / "df_final_3.parquet"
LOCAL_PARQUET_SYNTHETIC = PROJECT_ROOT / "data" / "processed" / "df_final_3_synthetic.parquet"
if USE_SYNTHETIC:
    LOCAL_PARQUET_PATH = LOCAL_PARQUET_SYNTHETIC
    PREFER_HIVE = False
elif LOCAL_PARQUET_DEFAULT.is_file():
    LOCAL_PARQUET_PATH = LOCAL_PARQUET_DEFAULT
    PREFER_HIVE = True
else:
    LOCAL_PARQUET_PATH = LOCAL_PARQUET_DEFAULT
    PREFER_HIVE = True
    print("[dataset] df_final_3.parquet нет — попробуем Hive (синтетика отключена)")
DATASET_PATH = LOCAL_PARQUET_PATH
RESULTS_DIR = PROJECT_ROOT / "integration" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CF_NAME = f"querulus_cf_{MODEL_VERSION}"
RG_NAME = f"querulus_rg_{MODEL_VERSION}"
print("MODEL_VERSION", MODEL_VERSION)
print("USE_SYNTHETIC", USE_SYNTHETIC, "| PREFER_HIVE", PREFER_HIVE)
print("LOCAL_PARQUET_PATH", LOCAL_PARQUET_PATH)


In [ ]:
from querulus.dataset.hadoop import load_df_final

# Приоритет: Hive. При ошибке Metastore/Spark — LOCAL_PARQUET_PATH (без синтетики, если USE_SYNTHETIC=False).
# Итог источника — блок «ИСТОЧНИК ДАТАСЕТА» в выводе load_df_final.
_df_raw, DATASET_SOURCE = load_df_final(
    hive_table=HIVE_TABLE,
    parquet_path=LOCAL_PARQUET_PATH,
    prefer_hive=PREFER_HIVE,
    generate_synthetic_if_missing=USE_SYNTHETIC,
)
if DATASET_SOURCE.startswith("hive:"):
    LOCAL_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
    _df_raw.to_parquet(LOCAL_PARQUET_PATH, index=False)
    print(f"[dataset] кэш после Hive записан в parquet: {LOCAL_PARQUET_PATH}")
else:
    print(f"[dataset] Hive не использован; данные из файла: {LOCAL_PARQUET_PATH}")

_df_raw = ensure_legacy_inflation_column(_df_raw)
if USE_SYNTHETIC:
    LOCAL_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
    _df_raw.to_parquet(LOCAL_PARQUET_PATH, index=False)

DATASET_PATH = LOCAL_PARQUET_PATH
df = dataframe_for_dsm(_df_raw)
print("df.shape", df.shape, "| DATASET_SOURCE", DATASET_SOURCE)
built = write_outboxml_configs(
    df,
    version=MODEL_VERSION,
    parquet_path=str(DATASET_PATH.as_posix()),
)
periods = built["periods"]
print("built configs", built["cf_path"].name, built["rg_path"].name)
print("periods table")
display(periods["table"])
print("отсечка Test_prod (prod_cutoff):", periods["prod_cutoff"])


In [ ]:
def _patch_dsm_models(dsm):
    for name, res in dsm.get_result().items():
        res.model = ensure_predictable_model(res.model)


def _prepared_X(dsm, model_name, data, *, ignore_row_filter=False):
    """Обёртка: признаки DSM. Для severity на полном Test — ignore_row_filter=True."""
    from querulus.training.outboxml_metrics import prepare_dsm_features

    return prepare_dsm_features(
        dsm, model_name, data, ignore_row_filter=ignore_row_filter
    )


def _predict_cf(dsm, model_name, data):
    from querulus.training.outboxml_metrics import predict_dsm_series

    return predict_dsm_series(
        dsm,
        model_name,
        data,
        task_type="classification",
        ignore_row_filter=False,
    )


def _predict_rg(dsm, model_name, data):
    """Severity на всех строках data (без фильтра TARGET_SEV > 0 из обучения)."""
    from querulus.training.outboxml_metrics import predict_dsm_series

    return predict_dsm_series(
        dsm,
        model_name,
        data,
        task_type="regression",
        ignore_row_filter=True,
    )


def _fin_effect_table(df_all, index, proba, sev, *, threshold=None, title=""):
    """Финэффект на заданном index; при threshold=τ — без повторного подбора порога."""
    cfg = resolve_fin_effect_config(
        df_all,
        frequency_target="TARGET_FREQ",
        severity_target="TARGET_SEV",
    )
    common = (
        pd.Index(index)
        .intersection(proba.dropna().index)
        .intersection(sev.dropna().index)
        .intersection(df_all.index)
    )
    if len(common) == 0:
        raise ValueError(
            "Нет пересечения index с proba/sev: проверьте index предсказаний."
        )
    coverage = len(common) / max(len(pd.Index(index)), 1)
    if coverage < 0.95:
        raise ValueError(
            f"pred покрывает только {len(common)}/{len(index)} строк ({coverage:.1%}). "
            "Для severity нужен predict без data_filter_condition "
            "(ignore_row_filter=True в _predict_rg)."
        )
    if len(common) < len(index):
        print(
            f"[fin_effect] строк с pred: {len(common)}/{len(index)} "
            f"(отброшено без proba/sev: {len(index) - len(common)})"
        )
    aligned = df_all.loc[common]
    fe = run_fin_effect_pipeline(
        aligned,
        proba.reindex(common),
        sev.reindex(common),
        aligned["TARGET_FREQ"],
        threshold=threshold,
        config=cfg,
    )
    if title:
        display(Markdown(f"### {title}"))
    print_best_threshold_report(fe)
    summary = fe.summary_table(cfg)
    display(summary.style.format("{:,.0f}", subset=summary.columns[3:], na_rep="—"))
    print(
        f"проверка Σ: model={summary['ФИН. ЭФФЕКТ МОДЕЛЬ'].sum():,.0f} "
        f"(отчёт {fe.model_effect_total:,.0f}), "
        f"fact={summary['ФИН. ЭФФЕКТ ФАКТ'].sum():,.0f} "
        f"(отчёт {fe.fact_effect_total:,.0f}), "
        f"экон.={summary['Экономия'].sum():,.0f} "
        f"(отчёт net {fe.net_effect:,.0f})"
    )
    n_pos = int((aligned["TARGET_FREQ"] == 1).sum())
    n_neg = int((aligned["TARGET_FREQ"] == 0).sum())
    print(f"выборка: n = {len(aligned)}, TARGET_FREQ=1: {n_pos}, TARGET_FREQ=0: {n_neg}")
    return fe, summary


## Parity-модели (DSM)

Обучение на parity train (см. `periods`). Модели не идут в прод; сравнение с блоком C3 `collect` — на Test.  
Порог τ frequency загружается из артефактов collect (`train_loop_new/`), в example не подбирается.


In [ ]:
from configs import config as querulus_outboxml_config
from querulus.fin_effect.threshold_policy import (
    load_collect_val_threshold as load_collect_threshold,
)
from querulus.training.dsm_fit import fit_dsm_classification

_collect_training = None
_tlr = globals().get("train_loop_result")
if _tlr is not None:
    _collect_training = getattr(_tlr, "training", None)

thr_collect = load_collect_threshold(
    PROJECT_ROOT,
    training=_collect_training,
)
print(f"τ frequency (collect) = {thr_collect:.2f}")

dsm_cf = DataSetsManager(
    config_name=str(built["cf_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["cf_path"]),
)
dsm_cf.load_dataset(data=df)
fit_dsm_classification(dsm_cf, CF_NAME, threshold=thr_collect)
_patch_dsm_models(dsm_cf)

dsm_rg = DataSetsManager(
    config_name=str(built["rg_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["rg_path"]),
)
dsm_rg.load_dataset(data=df)
dsm_rg.fit_models()
_patch_dsm_models(dsm_rg)


## Метрики parity: Train, Test, Test_prod

Одна обученная модель; колонки — Train (DSM train), Test (весь контрольный период), Test_prod (15% поздних дат Test).  
Frequency: метрики при τ из collect. Severity: строки с `TARGET_SEV > 0` (фильтр обучения RG).

In [ ]:
from querulus.training.outboxml_metrics import display_dsm_collect_metrics_cross_test

test_idx = periods["splits"].test
test_prod_idx = periods["prod_holdout_idx"]

for _dsm, _name, _task, _thr, _ignore_filter in (
    (dsm_cf, CF_NAME, "classification", thr_collect, False),
    (dsm_rg, RG_NAME, "regression", None, False),
):
    display_dsm_collect_metrics_cross_test(
        _dsm,
        _name,
        df,
        task_type=_task,
        val_threshold=_thr,
        test_slices={"test": test_idx, "test_prod": test_prod_idx},
        title=f"{_name}: train / test / test_prod",
        ignore_row_filter=_ignore_filter,
    )

## Финансовый эффект на Test

Предсказания на Test; τ фиксирован (из collect). Опционально — сравнение с `fin_effect_b` из `collect` (если ядро уже выполняло блок C3).


In [ ]:
_fe_collect = globals().get("fin_effect_b")
if _fe_collect is not None:
    display(Markdown("### Сравнение: collect C3 (Test)"))
    print_best_threshold_report(_fe_collect)
    _cfg_c = globals().get("FIN_EFFECT_CONFIG_B")
    if _cfg_c is not None:
        _sum_c = create_summary_table(_fe_collect.frame, _cfg_c)
        display(_sum_c.style.format("{:,.0f}", subset=_sum_c.columns[3:], na_rep="—"))

proba_test = _predict_cf(dsm_cf, CF_NAME, df.loc[test_idx])
sev_test = _predict_rg(dsm_rg, RG_NAME, df.loc[test_idx])

fe_parity, _ = _fin_effect_table(
    df,
    test_idx,
    proba_test,
    sev_test,
    threshold=thr_collect,
    title=f"Финэффект на Test (τ = {thr_collect:.2f})",
)

_fe_html = _fe_collect if _fe_collect is not None else fe_parity
_cfg_html = globals().get("FIN_EFFECT_CONFIG_B")
if _cfg_html is None:
    _cfg_html = resolve_fin_effect_config(
        df, frequency_target="TARGET_FREQ", severity_target="TARGET_SEV"
    )
_html_path = export_business_html(
    _fe_html,
    _cfg_html,
    path=PROJECT_ROOT / "notebooks" / "fin_effect_detailed.html",
    subtitle="Collect C3, Test" if _fe_collect is not None else "OutBoxML parity, Test",
)
print(f"HTML для бизнеса: {_html_path}")


## Prod-refit и финэффект на Test_prod

Обучение prod на train ∪ 85% Test до `prod_cutoff`. На **одном** срезе Test_prod (15% свежего test) — финэффект для двух моделей при том же τ из collect:

| Модель | Train |
|--------|-------|
| parity train | `parity_train` (укороченный train, как collect) |
| prod train | `prod_train_period` (train + 85% test до cutoff) |


In [ ]:
dsm_cf_prod = DataSetsManager(
    config_name=str(built["cf_prod_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["cf_prod_path"]),
)
dsm_cf_prod.load_dataset(data=df)
fit_dsm_classification(dsm_cf_prod, CF_NAME, threshold=thr_collect)
_patch_dsm_models(dsm_cf_prod)

dsm_rg_prod = DataSetsManager(
    config_name=str(built["rg_prod_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["rg_prod_path"]),
)
dsm_rg_prod.load_dataset(data=df)
dsm_rg_prod.fit_models()
_patch_dsm_models(dsm_rg_prod)

display_dsm_collect_metrics(
    dsm_cf_prod,
    CF_NAME,
    task_type="classification",
    val_threshold=thr_collect,
    title=f"prod {CF_NAME}",
)
display_dsm_collect_metrics(
    dsm_rg_prod, RG_NAME, task_type="regression", title=f"prod {RG_NAME}"
)

test_prod_idx = periods["prod_holdout_idx"]
print(
    f"Test_prod: n={len(test_prod_idx)}, "
    f"период {periods['prod_test_period'][0]} … {periods['prod_test_period'][1]}"
)

proba_parity_test_prod = _predict_cf(dsm_cf, CF_NAME, df.loc[test_prod_idx])
sev_parity_test_prod = _predict_rg(dsm_rg, RG_NAME, df.loc[test_prod_idx])
fe_parity_test_prod, _ = _fin_effect_table(
    df,
    test_prod_idx,
    proba_parity_test_prod,
    sev_parity_test_prod,
    threshold=thr_collect,
    title=f"Финэффект Test_prod — parity train (τ = {thr_collect:.2f})",
)

proba_prod_test_prod = _predict_cf(dsm_cf_prod, CF_NAME, df.loc[test_prod_idx])
sev_prod_test_prod = _predict_rg(dsm_rg_prod, RG_NAME, df.loc[test_prod_idx])
fe_prod_test_prod, _ = _fin_effect_table(
    df,
    test_prod_idx,
    proba_prod_test_prod,
    sev_prod_test_prod,
    threshold=thr_collect,
    title=f"Финэффект Test_prod — prod train (τ = {thr_collect:.2f})",
)

fe_prod = fe_prod_test_prod  # alias для export / email ниже

fe_test_prod_compare = pd.DataFrame(
    [
        {
            "train": "parity",
            "train_period": f"{periods['parity_train_period'][0]} … {periods['parity_train_period'][1]}",
            "n_test_prod": len(test_prod_idx),
            "net_effect": fe_parity_test_prod.net_effect,
            "model_effect": fe_parity_test_prod.model_effect_total,
            "fact_effect": fe_parity_test_prod.fact_effect_total,
        },
        {
            "train": "prod",
            "train_period": f"{periods['prod_train_period'][0]} … {periods['prod_train_period'][1]}",
            "n_test_prod": len(test_prod_idx),
            "net_effect": fe_prod_test_prod.net_effect,
            "model_effect": fe_prod_test_prod.model_effect_total,
            "fact_effect": fe_prod_test_prod.fact_effect_total,
        },
    ]
)
display(Markdown("### Сводка: финэффект на Test_prod (parity train vs prod train)"))
display(
    fe_test_prod_compare.style.format(
        {
            "net_effect": "{:,.0f}",
            "model_effect": "{:,.0f}",
            "fact_effect": "{:,.0f}",
        },
        na_rep="—",
    )
)
print(f"τ для сервиса (meta best_threshold): {thr_collect:.2f}")


## FactorsPlot, cohort и QuerulusEMailDSResult (prod)


In [ ]:
def _plot_features(dsm, model_name, *, n_num=6, n_cat=6):
    """Признаки для FactorsPlot из data_subset (в JSON targetslices пустые)."""
    subset = dsm.get_result()[model_name].data_subset
    nums = list(subset.features_numerical or [])[:n_num]
    cats = list(subset.features_categorical or [])[:n_cat]
    return nums + cats


def _show_figure(fig, title: str):
    if fig is None:
        print(f"[warn] {title}: figure is None")
        return
    show = getattr(fig, "show", None)
    if callable(show):
        show()
    else:
        display(fig)
    print(title, type(fig))


def _show_factors(export, model_name, features, *, bins=5):
    """По одной фиче: OutBoxML возвращает только последний figure из списка."""
    for feat in features:
        fig = export.plots(
            model_name=model_name,
            features=[feat],
            plot_type=1,
            bins_for_numerical_features=bins,
            use_exposure=False,
            only_test=True,
        )
        _show_figure(fig, f"FactorsPlot {model_name}: {feat}")


export_cf = ResultExport(ds_manager=dsm_cf_prod, config=querulus_outboxml_config)
export_rg = ResultExport(ds_manager=dsm_rg_prod, config=querulus_outboxml_config)
cf_plot_feats = _plot_features(dsm_cf_prod, CF_NAME)
rg_plot_feats = _plot_features(dsm_rg_prod, RG_NAME)
print("FactorsPlot features CF:", cf_plot_feats)
print("FactorsPlot features RG:", rg_plot_feats)

_show_factors(export_cf, CF_NAME, cf_plot_feats)
_show_factors(export_rg, RG_NAME, rg_plot_feats)

fig_cf_cohort = export_cf.plots(
    model_name=CF_NAME,
    plot_type=2,
    use_exposure=False,
    only_test=True,
    cut_min_value=0.1,
    cut_max_value=0.9,
    samples=100,
    cohort_base="model",
)
fig_rg_cohort = export_rg.plots(
    model_name=RG_NAME,
    plot_type=2,
    use_exposure=False,
    only_test=True,
    cut_min_value=0.1,
    cut_max_value=0.9,
    samples=100,
    cohort_base="model",
)
_show_figure(fig_cf_cohort, "Cohort plot_type=2 CF")
_show_figure(fig_rg_cohort, "Cohort plot_type=2 RG")

_prod_results = {}
_prod_results.update(dsm_cf_prod.get_result())
_prod_results.update(dsm_rg_prod.get_result())
try:
    QuerulusEMailDSResult(
        config=querulus_outboxml_config,
        ds_manager_result=_prod_results,
    ).success_mail(group_name=f"querulus_{MODEL_VERSION}")
    print("QuerulusEMailDSResult: письмо отправлено")
except Exception as exc:
    print(f"[warn] QuerulusEMailDSResult не отправлено: {type(exc).__name__}: {exc}")


## Экспорт артефактов для сервиса (prod)

Запись pickle frequency/severity и ансамбля, файла границ DQ, parquet `df_for_service` (исходный датасет с колонками `preds_cf` / `preds_rg`) и JSON с метаданными (`querulus_meta_*.json`), включая `best_threshold` = `thr_collect` (из collect).


In [ ]:
cf_export = dsm_cf_prod.get_result()[CF_NAME].dict_for_prod_export()
rg_export = dsm_rg_prod.get_result()[RG_NAME].dict_for_prod_export()
cf_export["model"] = ensure_predictable_model(cf_export["model"])
rg_export["model"] = ensure_predictable_model(rg_export["model"])

cf_pkl = RESULTS_DIR / f"querulus_cf_for_prod_{MODEL_VERSION}.pickle"
rg_pkl = RESULTS_DIR / f"querulus_rg_for_prod_{MODEL_VERSION}.pickle"
ans_pkl = RESULTS_DIR / f"querulus_ansamble_{MODEL_VERSION}.pickle"
dq_report = PROJECT_ROOT / "data" / "processed" / "data_quality_report.json"
dq_bounds_pkl = RESULTS_DIR / f"querulus_dq_bounds_{MODEL_VERSION}.json"
if dq_report.exists():
    write_service_dq_bounds(
        dq_bounds_pkl,
        model_version=MODEL_VERSION,
        report_path=dq_report,
    )
else:
    print("[warn] data_quality_report.json не найден — querulus_dq_bounds не записан")
    dq_bounds_pkl = None

cf_pkl.write_bytes(pickle.dumps([cf_export]))
rg_pkl.write_bytes(pickle.dumps([rg_export]))
ensemble = [deepcopy(cf_export), deepcopy(rg_export)]
ans_pkl.write_bytes(pickle.dumps(ensemble))

df_service = df.copy()
df_service["preds_cf"] = _predict_cf(dsm_cf_prod, CF_NAME, df)
df_service["preds_rg"] = _predict_rg(dsm_rg_prod, RG_NAME, df)
service_df_path = PROJECT_ROOT / "data" / "processed" / f"df_for_service_{MODEL_VERSION}.parquet"
df_service.to_parquet(service_df_path, index=True)

meta_path = RESULTS_DIR / f"querulus_meta_{MODEL_VERSION}.json"
meta = {
    "model_version": MODEL_VERSION,
    "periods": {
        k: list(v) if isinstance(v, tuple) else v
        for k, v in periods.items()
        if k in {
            "parity_train_period",
            "parity_test_period",
            "prod_train_period",
            "prod_test_period",
            "prod_cutoff",
            "date_column",
            "cal_period",
        }
    },
    "cf_name": CF_NAME,
    "rg_name": RG_NAME,
    "best_threshold": float(thr_collect),
    "calibration": None,
    "artifacts": {
        "cf": str(cf_pkl),
        "rg": str(rg_pkl),
        "ensemble": str(ans_pkl),
        "dq_bounds": str(dq_bounds_pkl) if dq_bounds_pkl else None,
        "df_for_service": str(service_df_path),
    },
    "preds_cf_col": "preds_cf",
    "preds_rg_col": "preds_rg",
}
meta_path.write_text(
    json.dumps(meta, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("=== Экспорт артефактов для сервиса (prod) ===")
print(f"  frequency (CF), pickle : {cf_pkl}")
print(f"  severity (RG), pickle    : {rg_pkl}")
print(f"  ансамбль CF+RG, pickle   : {ans_pkl}")
if dq_bounds_pkl is not None:
    print(f"  границы DQ, JSON         : {dq_bounds_pkl}")
print()
print("  df_for_service — parquet исходного df + колонки preds_cf / preds_rg:")
print(f"    путь      : {service_df_path}")
print(f"    shape     : {df_service.shape[0]:,} строк × {df_service.shape[1]} колонок")
print(f"    preds_cf  : доля NA = {df_service['preds_cf'].isna().mean():.1%}")
print(f"    preds_rg  : доля NA = {df_service['preds_rg'].isna().mean():.1%}")
print()
print("  метаданные — периоды, пути артефактов, τ frequency:")
print(f"    JSON      : {meta_path}")
print(f"    τ (collect, frequency best_threshold): {meta['best_threshold']:.2f}")
print("=== Готово ===")
